# makeshift — CPMG relaxation dispersion

End-to-end: from a set of CPMG planes to per-residue R₂,eff and a classification of which residues show conformational exchange (Rex).

In [ ]:
import matplotlib.pyplot as plt
from makeshift.utils.datasets import fetch
from makeshift.relaxation import CPMGExperiment

## 0. Get the example dataset

Downloads to `~/.makeshift/datasets/` and returns the data directory. The example config (`examples/SHP2_NSH2_CPMG.yaml`) already points at this location.

In [ ]:
data_dir = fetch('SHP2_NSH2_CPMG')
data_dir

## 1. Run the pipeline

`from_config` loads the config and reads the planes; `.run` picks peaks, maps the reference assignments, fits lineshapes, computes R₂,eff, classifies, and writes a results CSV. Pass `make_plots=True` for diagnostic PDFs. (~2 min on an M2.)

In [ ]:
exp = CPMGExperiment.from_config('../examples/SHP2_NSH2_CPMG.yaml').run('out/')
exp.classifications['label'].value_counts()

Results are stored on the experiment: `exp.ref_df` (picked + assigned peaks), `exp.all_r2eff` (per-peak dispersion), `exp.classifications` (labels + Rex).

In [ ]:
exp.classifications.head()

In [ ]:
# Estimated Rex per residue
c = exp.classifications
plt.errorbar(c['seqpos'], c['Rex_val'], yerr=c['Rex_err'], fmt='.')
plt.xlabel('Residue'); plt.ylabel('Rex (s⁻¹)')

## 2. Visualize

The relaxation plotting helpers take the results directly. Orange = conventional Rex, green = unsuppressed (elevated) R₂.

In [ ]:
from makeshift.relaxation.plotting import plot_waterfall, plot_r2eff_grid
plot_waterfall(exp.all_r2eff, exp.classifications, exp.ref_df)

In [ ]:
fig, axes = plot_r2eff_grid(exp.all_r2eff, exp.classifications, exp.ref_df)

## 3. Under the hood (optional)

`run()` is a thin wrapper over the stage functions. Call them directly to customize — e.g. tune the peak-mapping tolerance or supply your own `PeakList`.

In [ ]:
from makeshift import PeakList
from makeshift.spectra import Spectrum, map_peaklists
from makeshift.spectra.plotting import plot_spectrum, plot_peaklist
from makeshift.relaxation import load_config

config = load_config('../examples/SHP2_NSH2_CPMG.yaml')
hsqc = Spectrum.from_ucsf(config['reference'])
peaks = hsqc.pick_peaks(baseline=config['baseline_ref_plane'])
peaks.head()

Map a reference (assigned) peaklist onto the picked peaks. `map_peaklists` grid-searches the best translation, then does Hungarian matching. The reference can be a `PeakList`, a DataFrame, a BMRB id, or a CSV path — here the config gives a BMRB id.

In [ ]:
ref = PeakList.from_bmrb(config['peaklist']).data
assigned, ref_translated = map_peaklists(peaks, ref, tol=config['peak_mapping_tol'])

fig, ax = plot_spectrum(hsqc, baseline=config['baseline_ref_plane'],
                        xlim=(9.5, 7), ylim=(127, 112), cmap='Greys_r')
plot_peaklist(ax, peaks, text=None, color='tab:blue', label='picked')
plot_peaklist(ax, ref_translated, text='assn_label', color='tab:green', label='ref (translated)')
plot_peaklist(ax, assigned, text='assn_label', color='magenta', label='mapped')
plt.legend(loc='lower right')